In [1]:
%%capture

import os
os.environ["USE_TF"] = "0"

from transformers import TrainingArguments

import import_ipynb
from a_glance_at_dataset_and_tokenizer import xlmr_tokenizer, tags
from create_model import XLMRobertaForTokenClassification, tag_text, xlmr_config, device
from tokenizing_text import panx_de_encoded, xlmr_tokenizer
from performance_measures import align_predictions

In [2]:
num_epochs = 3
batch_size = 24
logging_steps = len(panx_de_encoded["train"]) // batch_size
xlmr_model_name = "xlm-roberta-base"
model_name = f"{xlmr_model_name}-finetuned-panx-de"

In [3]:
training_args = TrainingArguments(
    output_dir=model_name, log_level="error", num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_gpu_eval_batch_size=batch_size, eval_strategy="epoch",
    save_steps=1e6, weight_decay=0.01, disable_tqdm=False,
    logging_steps=logging_steps, push_to_hub=True,
)

In [4]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
from seqeval.metrics import f1_score

def compute_metrics(eval_pred):
    y_pred, y_true = align_predictions(eval_pred.predictions, eval_pred.label_ids)

    return {"f1": f1_score(y_true, y_pred)}

In [6]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)

In [7]:
def model_init():
    return (XLMRobertaForTokenClassification
           .from_pretrained(xlmr_model_name, config=xlmr_config)
           .to(device))

In [8]:
from transformers import Trainer

trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=panx_de_encoded["train"],
                  eval_dataset=panx_de_encoded["validation"],
                  processing_class=xlmr_tokenizer)

In [9]:
trainer.train()
trainer.push_to_hub(commit_message="Training Completed!")


2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/09/08 16:12:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/09/08 16:12:41 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/09/08 16:12:41 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Epoch,Training Loss,Validation Loss


NameError: name 'index2tag' is not defined

In [ ]:
text_de = "Jeff Dean ist ein Informatiker bei Google in Kalifornien"
tag_text(text_de, tags, trainer.model, xlmr_tokenizer)